In [ ]:
%%bash
# v3: the v2 hybrid trained on asl-signs train + ASL Citizen train (mapped signs) with webcam
# augmentation, three seeds, selected and calibrated on validation only, exported as fp16 ONNX.
# No test split is read here: the asl-signs and ASL Citizen test sets and test_clips_v2 are
# evaluated once, locally (kaggle/ASL_CITIZEN.md).
#
# Settings: Accelerator GPU (T4), Internet On, Save Version > Save & Run All (Commit).
# Inputs: Your Work > Datasets > edge-vision-src (the code; this notebook never clones the
# repository); Competitions > Google - Isolated Sign Language Recognition; and the ASL Citizen
# landmarks: Your Work > Datasets > asl-citizen-landmarks-private, or Your Work > Notebooks >
# asl_citizen_merge (the version whose log says COMPLETE).
# Resumable: finished seeds are skipped when an earlier version's output is added as an input.
# Python: Kaggle's own (3.13 when this was written) with its CUDA build of torch and the pinned
# versions of requirements.txt. If the training dependencies do not import there, a Python 3.11
# venv made with uv (torch of the CUDA build Kaggle ships, the same pins) is used instead.
# Licence: everything stays private (docs/DATA_LICENSES.md). Generated by
# scripts/build_kaggle_notebooks.py from kaggle/train_v3.sh.
set -u
NAME=train_v3
# ---- common to the edge-vision Kaggle notebooks (scripts/build_kaggle_notebooks.py) ----
# EV_WORK, EV_INPUT and EV_TMP exist for local tests only (scripts/kaggle_local_test.py).
WORK="${EV_WORK:-/kaggle/working}"
export EV_INPUT="${EV_INPUT:-/kaggle/input}"
TMPROOT="${EV_TMP:-/tmp}"
SRC="$TMPROOT/edge-vision-src"
# The fingerprint of the sources this notebook was built from; the code archive must match it.
SOURCE_FINGERPRINT="3db938681095f343"
VENV_PYTHON="3.11"
mkdir -p "$WORK" "$TMPROOT"
# clean runs a venv's Python with only the variables it needs. The Kaggle kernel exports its
# own: PYTHONPATH holds a sitecustomize that imports wrapt, and MPLBACKEND names
# matplotlib_inline, which made "import mediapipe" (it imports matplotlib.pyplot) fail in the
# venv. So the environment starts empty, keeps CLEAN_KEEP and every EV_* variable, and sets
# MPLBACKEND=Agg.
CLEAN_KEEP="HOME USER LOGNAME PATH LANG LC_ALL LC_CTYPE TZ TMPDIR TEMP TMP TERM
  LD_LIBRARY_PATH CUDA_VISIBLE_DEVICES CUDA_HOME NVIDIA_VISIBLE_DEVICES NVIDIA_DRIVER_CAPABILITIES
  OMP_NUM_THREADS HTTP_PROXY HTTPS_PROXY NO_PROXY http_proxy https_proxy no_proxy
  SSL_CERT_FILE SSL_CERT_DIR SYSTEMROOT WINDIR COMSPEC USERPROFILE APPDATA LOCALAPPDATA PATHEXT
  NUMBER_OF_PROCESSORS PROCESSOR_ARCHITECTURE"
clean() {
  local name keep=()
  for name in $CLEAN_KEEP $(compgen -e EV_); do
    if [ -n "${!name+x}" ]; then keep+=("$name=${!name}"); fi
  done
  env -i "${keep[@]}" MPLBACKEND=Agg PYTHONNOUSERSITE=1 "$@"
}
# Kaggle also sets UV_SYSTEM_PYTHON ("The --system flag has no effect" in uv venv's output).
uv() { env -u UV_SYSTEM_PYTHON -u VIRTUAL_ENV python -m uv "$@"; }
venv_python() { if [ -e "$1/bin/python" ]; then echo "$1/bin/python"; else echo "$1/Scripts/python.exe"; fi; }
kaggle_python() { python -c 'import platform; print(platform.python_version())' 2>&1; }
# find below the inputs without walking the asl-signs parquet tree or the landmark shards
find_inputs() { find "$EV_INPUT" \( -name train_landmarks -o -name shards \) -prune -o "$@" -print 2>/dev/null | sort; }

# make_venv DIR: an empty Python $VENV_PYTHON venv, made with uv (Kaggle's Python has no
# ensurepip, and no mediapipe 0.10.21 wheel exists for it).
make_venv() {
  echo "--- pip install uv"
  python -m uv --version 2>/dev/null || python -m pip install -q --disable-pip-version-check uv \
    || { echo "STEP FAILED: pip install uv"; return 1; }
  echo "--- uv python install $VENV_PYTHON"
  uv python install "$VENV_PYTHON" || { echo "STEP FAILED: uv python install $VENV_PYTHON"; return 1; }
  echo "--- uv venv --python $VENV_PYTHON $1"
  uv venv --clear --python "$VENV_PYTHON" "$1" || { echo "STEP FAILED: uv venv $1"; return 1; }
}

# The code comes ONLY from the private dataset edge-vision-src: edge-vision-src.tar.gz, made by
# "python scripts/build_kaggle_notebooks.py --source-archive", or that archive unpacked by Kaggle.
get_source() {
  local archives archive tree found
  rm -rf "$SRC"
  archives=$(find_inputs -type f -name 'edge-vision-src*.tar.gz')
  archive=$(echo "$archives" | head -1)
  tree=$(find_inputs -type f -name SOURCE_COMMIT | head -1)
  if [ "$(echo "$archives" | grep -c .)" -gt 1 ]; then
    echo "WARNING: several code archives are attached; using the first:"
    echo "$archives"
  fi
  if [ -n "$archive" ]; then
    echo "code: $archive"
    python -c 'import sys, tarfile; tarfile.open(sys.argv[1]).extractall(sys.argv[2], filter="data")' \
      "$archive" "$SRC" || { echo "CODE SOURCE UNREADABLE: could not unpack $archive"; return 1; }
  elif [ -n "$tree" ]; then
    echo "code: $(dirname "$tree") (the archive, unpacked by Kaggle)"
    cp -r "$(dirname "$tree")" "$SRC" \
      || { echo "CODE SOURCE UNREADABLE: could not copy $(dirname "$tree")"; return 1; }
  else
    echo "CODE SOURCE MISSING: no edge-vision-src.tar.gz under $EV_INPUT."
    echo "  These notebooks never clone the repository. Upload artifacts/v3/upload/edge-vision-src.tar.gz"
    echo "  (python scripts/build_kaggle_notebooks.py --source-archive) as a new Private dataset named"
    echo "  edge-vision-src, add it here with Add Input > Your Work > Datasets, and run again."
    return 1
  fi
  if [ ! -f "$SRC/SOURCE_COMMIT" ]; then
    echo "CODE SOURCE INVALID: no SOURCE_COMMIT in it; build the archive with --source-archive"
    return 1
  fi
  echo "code commit: $(cat "$SRC/SOURCE_COMMIT")"
  found=$(python "$SRC/scripts/build_kaggle_notebooks.py" --fingerprint 2>&1)
  if [ "$found" != "$SOURCE_FINGERPRINT" ]; then
    echo "CODE SOURCE MISMATCH: this notebook was built from notebook sources $SOURCE_FINGERPRINT,"
    echo "  the edge-vision-src input holds '$found'. Use the archive and the notebooks of one commit:"
    echo "  upload that commit's edge-vision-src.tar.gz as a new version of the dataset, or import"
    echo "  the notebooks again from the commit the archive was made from."
    return 1
  fi
  echo "code fingerprint: $found (matches this notebook)"
}
# ---- end of the common part ----
LOG="$WORK/train_v3.log"
MAX_HOURS=9              # no new seed starts after this, so export and packing fit in 12 h
START=$(date +%s)
TRAINENV="$TMPROOT/trainenv"
USE_VENV=0
CHOICE="$TMPROOT/train_python"   # which Python the block chose, for the packing step after it
rm -f "$CHOICE"
# py runs the training code with the chosen Python: Kaggle's, or the uv venv (USE_VENV=1).
py() { if [ "$USE_VENV" = 1 ]; then clean "$(venv_python "$TRAINENV")" "$@"; else python "$@"; fi; }
versions() {
  py - <<'PYEOF'
import importlib, platform
print(f"python {platform.python_version()}")
for name in ("torch", "numpy", "pandas", "pyarrow", "pydantic", "yaml", "psutil", "onnx", "onnxruntime"):
    try:
        module = importlib.import_module(name)
        extra = ""
        if name == "torch":
            extra = f" (cuda {module.version.cuda}, available {module.cuda.is_available()})"
        print(f"{name} {getattr(module, '__version__', '?')}{extra}")
    except Exception as error:
        print(f"{name} MISSING: {type(error).__name__}: {error}")
PYEOF
}
# Every package the training code imports loads, with the repository, and torch runs an op.
deps_ok() {
  py -c "import torch, numpy, pandas, pyarrow, pydantic, yaml, psutil, onnx, onnxruntime; import edgevision.train.data, edgevision.data.citizen; torch.zeros(2).sum().item(); print('training dependencies: ok')"
}
# The fallback: a Python 3.11 venv with the torch build Kaggle ships (PyPI's if that fails).
train_venv() {
  local vpy torch_version installed=""
  make_venv "$TRAINENV" || return 1
  vpy=$(venv_python "$TRAINENV")
  torch_version=$(python -c 'import torch; print(torch.__version__)' 2>/dev/null)
  case "$torch_version" in
    *+cu*)
      echo "--- uv pip install torch==${torch_version%%+*} (the ${torch_version##*+} build Kaggle ships)"
      uv pip install --python "$vpy" "torch==${torch_version%%+*}" \
        --index-url "https://download.pytorch.org/whl/${torch_version##*+}" && installed=1
      ;;
  esac
  if [ -z "$installed" ]; then
    echo "--- uv pip install torch (PyPI)"
    uv pip install --python "$vpy" torch || { echo "STEP FAILED: uv pip install torch"; return 1; }
  fi
  echo "--- uv pip install -r requirements.txt onnx==1.23.2 onnxruntime==1.31.1"
  uv pip install --python "$vpy" -r requirements.txt "onnx==1.23.2" "onnxruntime==1.31.1" \
    || { echo "STEP FAILED: uv pip install of the pinned requirements"; return 1; }
  uv pip install --python "$vpy" --no-deps -e . || { echo "STEP FAILED: installing the repository"; return 1; }
}
{
  echo "=== $(date -u '+%F %T') UTC: start $NAME; Kaggle python $(kaggle_python)"
  get_source || { echo "STOPPED: no usable code source (see above); nothing was trained."; exit 0; }
  cd "$SRC" || exit 0
  echo "--- versions in Kaggle's environment"
  versions
  # Kaggle's CUDA torch stays; the other packages are pinned to the versions tested locally.
  python -m pip install -q --disable-pip-version-check -r requirements.txt "onnx==1.23.2" \
    || echo "WARNING: pip could not install the pinned requirements on Kaggle's Python"
  python -c "import onnxruntime" 2>/dev/null \
    || python -m pip install -q --disable-pip-version-check "onnxruntime==1.31.1"
  python -m pip install -q --disable-pip-version-check --no-deps -e . \
    || echo "WARNING: pip install of the repository failed on Kaggle's Python"
  echo "--- dependency check on Kaggle's Python $(kaggle_python)"
  if ! deps_ok; then
    echo "Kaggle's Python cannot run the training code (error above); making a Python $VENV_PYTHON venv"
    if ! train_venv; then
      echo "ENVIRONMENT FAILED: neither Kaggle's Python nor the uv venv can run the training code;"
      echo "the full error is above. Nothing was trained."
      exit 0
    fi
    USE_VENV=1
    echo "--- dependency check in $TRAINENV"
    deps_ok || { echo "ENVIRONMENT FAILED: the venv cannot import the training dependencies (above); nothing was trained."; exit 0; }
  fi
  echo "$USE_VENV" > "$CHOICE"
  echo "--- versions used for training"
  versions
  py -c "import torch; print('GPU', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)); print('kernel check', (torch.ones(4, device='cuda') * 2).sum().item())" \
    || echo "WARNING: no usable GPU; training on the CPU is far too slow"

  # The inputs: asl-signs, the ASL Citizen landmarks (dataset or merge-run output, at any depth,
  # zipped or not; smoke-test and unfinished outputs are skipped) and finished runs of an earlier
  # version of this notebook. locate lists everything it finds.
  echo "--- inputs under $EV_INPUT"
  DATA_ROOT=""; LANDMARKS=""; PREVIOUS_RUNS=()
  eval "$(py scripts/v3_kaggle.py locate --input "$EV_INPUT" --unpack-dir "$TMPROOT")"
  echo "asl-signs: ${DATA_ROOT:-NOT FOUND}"
  echo "ASL Citizen landmarks: ${LANDMARKS:-NOT FOUND}"
  if [ -z "$DATA_ROOT" ] || [ -z "$LANDMARKS" ]; then
    echo "ERROR: an input is missing (see the list above); add it with Add Input and commit again; stopping."
    exit 0
  fi
  for previous in "${PREVIOUS_RUNS[@]}"; do
    mkdir -p "$WORK/runs" && cp -rn "$previous" "$WORK/runs/" && echo "carried over runs from $previous"
  done

  if [ ! -f "$WORK/processed_v2/features.npz" ]; then
    py scripts/preprocess.py --config experiments/preprocess_v2.yaml --data-root "$DATA_ROOT" --output-dir "$WORK/processed_v2"
  fi
  # The asl-signs split must hold the participants of the shipped v2 model, so v2 and v3 are
  # compared on the same test sequences.
  py -c "import json; a = json.load(open('$WORK/processed_v2/splits_signer_independent.json'))['participants']; b = json.load(open('results/runs/cnn_signer_independent/seed_0/splits_signer_independent.json'))['participants']; assert a == b, 'split differs from v1/v2'; print('asl-signs split: same participants as v1/v2')" \
    || { echo "ERROR: the asl-signs split differs; stopping"; exit 0; }
  if [ ! -f "$WORK/processed_v3/features.npz" ]; then
    py scripts/v3_kaggle.py build-dataset --landmarks "$LANDMARKS" --kaggle-processed "$WORK/processed_v2" --out "$WORK/processed_v3"
  fi

  for SEED in 0 1 2; do
    if [ -f "$WORK/runs/e_hybrid_v3citizen/seed_$SEED/summary.json" ]; then
      echo "seed $SEED: done"
      continue
    fi
    ELAPSED=$(( ($(date +%s) - START) / 3600 ))
    if [ "$ELAPSED" -ge "$MAX_HOURS" ]; then
      echo "seed $SEED: not started after $ELAPSED h; save this version and rerun with its output attached"
      continue
    fi
    py scripts/train.py --config experiments/train_e_hybrid_v3citizen.yaml --seed "$SEED" \
      --processed-dir "$WORK/processed_v3" --output-root "$WORK/runs"
  done

  if CHOSEN=$(py scripts/v3_kaggle.py select --runs "$WORK/runs/e_hybrid_v3citizen" --out "$WORK/v3_export/selection.json"); then
    echo "chosen on validation: $CHOSEN"
    py scripts/export_web.py --run "$CHOSEN" --processed-dir "$WORK/processed_v3" --variant fp16 \
      --data-source asl-signs+asl-citizen --out "$WORK/v3_export/model" \
      --golden-dir "$WORK/v3_export/golden" --work-dir "$WORK/v3_export/work"
  fi
  echo "=== $(date -u '+%F %T') UTC: packing"
} 2>&1 | tee -a "$LOG"
# The block ran in a subshell (the pipe); its choice of Python is read back from $CHOICE.
if [ -f "$CHOICE" ]; then
  USE_VENV=$(cat "$CHOICE")
  (cd "$SRC" && py scripts/v3_kaggle.py pack --work "$WORK" --output "$WORK/v3_bundle.zip") 2>&1 | tee -a "$LOG"
fi
echo "=== $(date -u '+%F %T') UTC: finished; this cell always exits 0, so read $LOG" | tee -a "$LOG"
exit 0
